# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR^2 dataset) Exploration with `mlcroissant`

This notebook demonstrates loading and exploration of a clinicopathology dataset using the `mlcroissant` library. All Croissant entities (record sets, fields, columns) are referenced via their `@id` to ensure semantic clarity and reproducibility.

### Dataset Source

The dataset is described by a [Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load both the dataset Croissant metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load Croissant dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset loaded: {metadata.name}\n\nDescription: {metadata.description}\n\nIdentifier: {getattr(metadata, 'identifier', 'n/a')}")

## 2. Data Overview

Let's list all available Record Sets, their `@id`s, and their fields, so you can reference them easily.

> **Note:** All entities are referenced by their `@id` (the standard Croissant identifier).

In [ ]:
# List all record sets with their @id and (if possible) short metadata info and all fields' @ids.
recordsets = list(dataset.record_sets())

for rs in recordsets:
    print(f"RecordSet: {rs['@id']}")
    # Try to print name/description if present
    name = rs.get('name', '(no name)')
    desc = rs.get('description', '')
    print(f"  Name: {name}")
    print(f"  Description: {desc}")
    # List fields by @id
    fields = rs.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    print(f"  Fields:")
    for f in fields:
        # field may be dict or string (id)
        fid = f['@id'] if isinstance(f, dict) and '@id' in f else (f if isinstance(f, str) else 'unknown')
        print(f"    - {fid}")
    print()

## 3. Data Extraction

Let's select the main tabular record set for analysis and extract it to a DataFrame.

**Tip:** Use the `@id` of the record set from the cell above (`recordset_id`) and list of fields for the DataFrame columns.

In [ ]:
# Step 1: Select a record set for extraction (replace with your @id from above as applicable)
# For this dataset, let's assume the main table's @id is as example below.
# You can copy and paste an actual @id from the previous output if this is incorrect.
main_recordset_id = None
for rs in dataset.record_sets():
    if 'clinicopathological' in str(rs.get('name', '')).lower() or 'colorectal' in str(rs.get('name', '')).lower():
        main_recordset_id = rs['@id']
        break
if main_recordset_id is None:
    # Default - just take the first
    main_recordset_id = dataset.record_sets()[0]['@id']

# Step 2: Extract records from all tabular record sets (most datasets just have one)
record_set_ids = [rs['@id'] for rs in dataset.record_sets()]
dataframes = {}

for rsid in record_set_ids:
    records_list = list(dataset.records(record_set=rsid))
    if records_list:
        dataframes[rsid] = pd.DataFrame(records_list)

print(f"Loaded DataFrames for record sets: {list(dataframes.keys())}")

# Let's look at the columns for the main record set
if main_recordset_id in dataframes:
    print(f"Columns in main record set ({main_recordset_id}):\n", dataframes[main_recordset_id].columns.tolist())
    display(dataframes[main_recordset_id].head())
else:
    print(f"Main record set {main_recordset_id} not found in loaded DataFrames.")

## 4. Exploratory Data Analysis (EDA)

We'll now explore data operations on this record set using some numeric and categorical fields.

- We'll select a numeric field via its `@id`, e.g. `'age_at_second_crc_dx'` (replace with the actual field `@id` from your schema).
- We'll filter for patients above a threshold age.
- We'll normalize the numeric field, and group by an anatomical or categorical field such as `'anatomical_location'`.

In [ ]:
import numpy as np
df = dataframes.get(main_recordset_id)

# Replace these with actual @id's (column names) as found in the DataFrame
numeric_field_id = None
for c in df.columns:
    if 'age' in c.lower():
        numeric_field_id = c
        break
if numeric_field_id is None:
    numeric_field_id = df.select_dtypes(include=np.number).columns[0]

print(f"Using numeric field: {numeric_field_id}")

threshold = 60 # e.g., filter patients older than 60

filtered_df = df.copy()
filtered_df = filtered_df[pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold} (count: {filtered_df.shape[0]}):")
display(filtered_df.head())

# Normalization
norm_col = f"{numeric_field_id}_normalized"
filtered_df[norm_col] = (
    pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()
) / pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()

print(f"\nNormalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, norm_col]].head())

# Grouping by a categorical field (e.g., anatomical location)
group_field_id = None
for c in df.columns:
    if 'anatomical' in c.lower() or 'sex' in c.lower() or 'location' in c.lower():
        group_field_id = c
        break

if group_field_id:
    print(f"\nGrouped mean of {numeric_field_id}_normalized by {group_field_id}:")
    grouped_df = filtered_df.groupby(group_field_id)[norm_col].mean()
    display(grouped_df)
else:
    print("No suitable group field found for grouping.")

## 5. Visualization

Visualize the data distribution (e.g., age distribution, anatomical location counts) in the cohort.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Age histogram
plt.figure(figsize=(8,5))
sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce'), bins=15, kde=True)
plt.xlabel(numeric_field_id)
plt.title('Distribution of ' + numeric_field_id)
plt.show()

# Anatomical location barplot if present
if group_field_id:
    plt.figure(figsize=(8,4))
    sns.countplot(data=df, x=group_field_id, order=df[group_field_id].value_counts().index)
    plt.title(f'Distribution of {group_field_id}')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

- We successfully loaded the FAIR^2 Clinicopathological Colorectal Cancer Dataset via its Croissant schema.
- All tabular record sets, fields, and columns were referenced by their unique `@id`.
- A preliminary analysis showed the age distribution and provided group-level statistics by anatomical location (or other fields as available).
- You can now proceed to deeper statistical or ML analysis using these DataFrames, always referencing data elements by their `@id` for reproducibility.